# Consumo ilimitado: a fatura que ninguém esperava

OWASP Top 10 para LLMs 2025 — **LLM10 Unbounded Consumption**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Terceiros descobrem o endpoint do Aurora Assist e passam a usá-lo como tradutor/gerador de texto gratuito. Cada chamada custa um centavo simulado; sem limite, o volume de terceiros domina o custo do dia.

In [2]:
CUSTO_POR_CHAMADA = 0.015  # reais, valor simulado

def simular_trafego(chamadas, limitador=None):
    custo_total = 0.0
    bloqueadas = 0
    for origem in chamadas:
        if limitador is not None and not limitador.permitir(origem):
            bloqueadas += 1
            continue
        custo_total += CUSTO_POR_CHAMADA
    return custo_total, bloqueadas

# 20 chamadas legítimas de alunos + 300 chamadas de um terceiro não identificado
trafego = ["aluno"] * 20 + ["terceiro-desconhecido"] * 300


## Sem defesa

In [3]:
custo_vulneravel, _ = simular_trafego(trafego)
print("custo do dia sem limite: R$ {:.2f} ({} chamadas)".format(custo_vulneravel, len(trafego)))
assert custo_vulneravel > 4.0


custo do dia sem limite: R$ 4.80 (320 chamadas)


## Com defesa
`defesas.LimitadorTaxa` limita quantas chamadas cada origem pode fazer.

In [4]:
limitador = defesas.LimitadorTaxa(maximo=5)
custo_defendido, bloqueadas = simular_trafego(trafego, limitador)
print("custo do dia com limite: R$ {:.2f} ({} chamadas bloqueadas)".format(custo_defendido, bloqueadas))
assert custo_defendido < custo_vulneravel


custo do dia com limite: R$ 0.15 (310 chamadas bloqueadas)


## O que observar
- Cada chamada ao modelo custa dinheiro de verdade, diferente de uma consulta comum a um banco de dados — por isso consumo é, aqui, um recurso de segurança.
- O ataque chamado "denial of wallet" não tira o serviço do ar: ele deixa o serviço no ar e explode o custo, o que pode passar despercebido até a fatura chegar.
- Limite de taxa por chave/origem é o controle mais barato de ligar no primeiro dia; quota por usuário, teto de tokens por requisição e alerta de custo compõem o resto da defesa.